# Worked example: choosing an approach on 500 labelled reviews

You have 500 labelled movie-review sentences and must decide how to classify new ones. The method is the same one Lab 09 asks for: build a sensible baseline, try the pretrained approaches, **choose on the validation split**, and score the test split **once**.

The full run takes about eight minutes on a CPU, so the final script is `compare_methods.py`. This notebook walks through the same steps in pieces.

In [1]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

test_texts, test_labels = as_lists(reviews["test"])
texts, labels = small_sample(reviews["train"], 500, seed=42)
len(texts), sum(labels), len(val_texts), len(test_texts)

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(500, 250, 1066, 1066)

In [2]:
from baselines import tfidf_baseline

baseline = tfidf_baseline(texts, labels, val_texts, val_labels)
baseline

{'accuracy': 0.636, 'macro_f1': 0.636}

In [ ]:
from sklearn.linear_model import LogisticRegression

from baselines import scores
from embed import embed_texts

features = LogisticRegression(max_iter=2000).fit(embed_texts(texts), labels)
frozen = scores(val_labels, features.predict(embed_texts(val_texts)))
frozen

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 481.14it/s]


The two fine-tuning runs take a few minutes each, so run them from the script, then read `worked_metrics_seed42.json`:

    python compare_methods.py 42

The script trains all four approaches, prints their validation accuracy, picks the best, and scores only that one on the test split. Run it with another seed and `--validation-only` to see how much the validation numbers move without touching the test split:

    python compare_methods.py 1 --validation-only

In [ ]:
import json
from pathlib import Path

path = Path("worked_metrics_seed42.json")
if not path.exists():
    path = Path("reference_results/worked_metrics_seed42.json")  # saved reference run
print(json.dumps(json.loads(path.read_text()), indent=2))